# 06 — Social (publication charts)

The owner reviewed `04-viz` and confirmed the framing, so this notebook renders the publication-ready set. Four charts, each in **two targets** from the same `chart_` tables:
- **Social** — full chrome (title, subtitle, source, watermark), `twitter_landscape` (1600×900) → `outputs/social/`.
- **Web** — `web_mode=True` (drops title/subtitle/source, keeps only the `@unwelcomedata` watermark), `web` preset (1664×936) → `outputs/web/`. These get embedded on the Pages site, which supplies its own headings.

**The four charts (confirmed in `04-viz`):**
1. **Video game critic review scores** — distribution of IGDB critic ratings (hero); teal bars, 90+ band aqua.
2. **Video game user review scores** — distribution of IGDB user ratings; **same colours**, the critic-vs-user distinction is carried in the title.
3. **Average critic vs user rating by year** — two lines (critic teal, user caramel).
4. **IGDB user rating vs critic rating** — scatter; the 12 most visually-isolated games coloured rust and labelled.

⚠️ **Web info-preservation:** web mode drops the title/subtitle/source, so any fact living ONLY there must be restated in the Pages README above each chart:
- Charts 1 & 2 differ ONLY by **critic vs user** (title) — in web mode the two histograms look near-identical, so each needs its own labelled heading. (The x-axis labels “IGDB critic rating” / “IGDB user rating” do survive in-image.)
- Chart 3: the metric (**mean rating per year**), the year range, and the ≥10-games filter live in the subtitle → restate above the chart. The **Critic/User** line end-labels survive in-image.
- Chart 4: **what ‘outlier’ means** (most isolated), the r value, and the dropped critic=0 row live in the subtitle → restate. Axis labels + point labels survive.

*Titles are **descriptive** (house default). Source = IGDB. Read-only DuckDB; closed in the Cleanup cell.*

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

import duckdb
from IPython.display import display
from src.ingest import load_config
from colors import c
from chart_templates import histogram, line_chart, scatter_plot
from viz import PRESETS

cfg = load_config('config.yaml')
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
soc_w, soc_h, _ = PRESETS['twitter_landscape']   # 1600 x 900
web_w, web_h, _ = PRESETS['web']                 # 1664 x 936
social_out = Path(cfg['paths']['outputs_social']); social_out.mkdir(parents=True, exist_ok=True)
web_out = social_out.parent / 'web'; web_out.mkdir(parents=True, exist_ok=True)
SRC = 'IGDB (Internet Game Database) API'
print('presets  social', (soc_w, soc_h), ' web', (web_w, web_h))

## Chart 1 (hero) — Video game critic review scores

Distribution of IGDB critic ratings across the 6,005 games with ≥ 3 critic scores. 5-point bands, mean + median reference lines, 90+ club highlighted aqua.

In [ ]:
df_all = con.execute('SELECT critic_rating, user_rating FROM chart_critic_all').df()
df_user = df_all[df_all['user_rating'].notna()]
n_critic = len(df_all)

def critic_hist(title, subtitle, source, w, h, web):
    return histogram(
        df_all, value_col='critic_rating',
        bin_edges=list(range(0, 101, 5)), x_range=(0, 100), x_tick_step=10,
        x_axis_label='IGDB critic rating', y_axis_label='Number of games',
        bar_color=c('teal'), highlight_range=(90, 100, c('aqua')),
        title=title, subtitle=subtitle, source=source,
        img_width=w, img_height=h, web_mode=web)

img = critic_hist(
    'Video game critic review scores',
    f'Distribution of IGDB critic ratings — {n_critic:,} games (≥ 3 critic scores each); each bar = a 5-point band. 90+ highlighted.',
    SRC, soc_w, soc_h, False)
img.save(social_out / '01_critic_score_distribution.png'); display(img)
critic_hist('', None, None, web_w, web_h, True).save(web_out / '01_critic_score_distribution.png')
print('saved 01 (social + web)')

## Chart 2 — Video game user review scores

The same histogram treatment for the community/user rating — **same colours** as the critic hero so the pair reads as one set; the distinction is in the title. Fewer games (only those with a user rating).

In [ ]:
n_user = len(df_user)

def user_hist(title, subtitle, source, w, h, web):
    return histogram(
        df_user, value_col='user_rating',
        bin_edges=list(range(0, 101, 5)), x_range=(0, 100), x_tick_step=10,
        x_axis_label='IGDB user rating', y_axis_label='Number of games',
        bar_color=c('teal'), highlight_range=(90, 100, c('aqua')),
        title=title, subtitle=subtitle, source=source,
        img_width=w, img_height=h, web_mode=web)

img = user_hist(
    'Video game user review scores',
    f'Distribution of IGDB user ratings — {n_user:,} games with a community rating; each bar = a 5-point band. 90+ highlighted.',
    SRC, soc_w, soc_h, False)
img.save(social_out / '02_user_score_distribution.png'); display(img)
user_hist('', None, None, web_w, web_h, True).save(web_out / '02_user_score_distribution.png')
print('saved 02 (social + web)')

## Chart 3 — Average critic vs user rating by year

One line for the mean **critic** rating, one for the mean **user** rating, per release year (from `chart_avg_by_year`: years with ≥ 10 scored games, cut at the last complete year). The lines cross around 2010–11 — users rated higher in the 2000s, critics pull ahead in the 2010s.

In [ ]:
avg = con.execute('SELECT year, avg_critic, avg_user FROM chart_avg_by_year ORDER BY year').df()
yr_lo, yr_hi = int(avg['year'].min()), int(avg['year'].max())

def year_line(title, subtitle, source, w, h, web):
    return line_chart(
        avg, x_col='year',
        series=[{'col': 'avg_critic', 'label': 'Critic', 'color': c('teal')},
                {'col': 'avg_user',   'label': 'User',   'color': c('caramel')}],
        x_axis_label='Release year', y_axis_label='Average rating (0–100)',
        y_min=60, y_max=90, markers=True, label_last=True,
        title=title, subtitle=subtitle, source=source,
        img_width=w, img_height=h, web_mode=web)

img = year_line(
    'Video game review scores — average critic vs user rating by year',
    f'IGDB ratings, mean per release year ({yr_lo}–{yr_hi}). Years with ≥ 10 scored games.',
    SRC, soc_w, soc_h, False)
img.save(social_out / '03_avg_rating_by_year.png'); display(img)
year_line('', None, None, web_w, web_h, True).save(web_out / '03_avg_rating_by_year.png')
print('saved 03 (social + web)')

## Chart 4 — IGDB user rating vs critic rating (outliers labelled)

Point per game (from `chart_scatter`: both ratings, critic > 0). A dashed y = x agreement line; the 12 most **visually isolated** games (most empty space around them, by nearest-neighbour distance) are coloured rust and labelled. The `is_outlier` flag and the outlier definition are baked into `03-prepare`, so this reads straight from the table.

In [ ]:
sc = con.execute('SELECT name, critic_rating, user_rating, is_outlier FROM chart_scatter').df()
corr = sc['critic_rating'].corr(sc['user_rating'])
n_out = int(sc['is_outlier'].sum())

def rating_scatter(title, subtitle, source, w, h, web):
    return scatter_plot(
        sc, x_col='critic_rating', y_col='user_rating',
        x_axis_label='IGDB critic rating', y_axis_label='IGDB user rating',
        point_color=c('teal'),
        label_col='name', outlier_col='is_outlier',
        outlier_color=c('spice'), label_outliers_only=True,
        ref_lines=[{'kind': 'diagonal', 'color': c('gray'), 'label': 'users = critics'}],
        title=title, subtitle=subtitle, source=source,
        img_width=w, img_height=h, web_mode=web)

img = rating_scatter(
    'Video game review scores — IGDB user rating vs critic rating',
    f'{len(sc):,} games with both ratings (0–100). Pearson r = {corr:.2f}. Highlighted = the {n_out} most isolated games (most empty space around them).',
    SRC, soc_w, soc_h, False)
img.save(social_out / '04_user_vs_critic_scatter.png'); display(img)
rating_scatter('', None, None, web_w, web_h, True).save(web_out / '04_user_vs_critic_scatter.png')
print('saved 04 (social + web)')

---
**Next:** run `scripts/validate_charts.py` (must exit 0) — it re-derives every chart's facts from DuckDB, checks the export matches, and confirms social/web parity. Then the project is ready for the independent validation pass and fun-tier release curation (`public-release.md`).

---
## Cleanup
Close the read-only DuckDB connection so the lock is released.

In [ ]:
con.close()
print('connection closed')